In [ ]:
import requests
import os
from pathlib import Path

data_dir = Path("data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

taxi_file = data_dir / "taxi_data.parquet"
zone_file = data_dir / "taxi_zone_lookup.csv"

# Only download taxi data if not already present
if not taxi_file.exists():
    url1 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
    response1 = requests.get(url1)
    response1.raise_for_status()
    taxi_file.write_bytes(response1.content)
    print("Taxi data downloaded.")
else:
    print("Taxi data already exists. Skipping download.")

# Only download zone lookup if not already present
if not zone_file.exists():
    url2 = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
    response2 = requests.get(url2)
    response2.raise_for_status()
    zone_file.write_bytes(response2.content)
    print("Zone lookup downloaded.")
else:
    print("Zone lookup already exists. Skipping download.")

print("Download check complete!")

In [ ]:

import polars as pl
import sys
from polars import Datetime

try:
        df = pl.read_parquet("data/raw/taxi_data.parquet")
        required_cols = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID","DOLocationID",
                        "passenger_count", "trip_distance", "fare_amount", "tip_amount", "total_amount", "payment_type"]
        
        missing = [col for col in required_cols if col not in df.columns]
        if missing:
            raise ValueError(f"Missing columns: {missing}")
            
        date_cols = ("tpep_pickup_datetime", "tpep_dropoff_datetime")
        for col in date_cols:
            if df.schema[col] != pl.Datetime:
                raise TypeError(f"{col} is not of type Datetime")

        print(f"Number of rows in the dataframe: {len(df):,}")

except Exception as e:
    print(f"Error loading data: {e}")

In [ ]:
import polars as pl

def apply_filter(df, condition, message):
    start_rows = df.height
    df = df.filter(condition)
    removed = start_rows - df.height
    print(f"{message}: {removed:,}")
    return df

# Initial row count
print(f"Starting rows: {df.height:,}")

# Drop nulls
start_rows = df.height
df = df.drop_nulls(subset=[
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "fare_amount"
])
print(f"Rows removed with nulls: {start_rows - df.height:,}")

# Sequential filters
df = apply_filter(df, pl.col('passenger_count') > 0,
                  "Rows removed where passenger_count <= 0")

df = apply_filter(df, pl.col('trip_distance') > 0,
                  "Rows removed where trip_distance <= 0")

df = apply_filter(df, pl.col('fare_amount') > 0,
                  "Rows removed where fare_amount <= 0")

df = apply_filter(df, pl.col('fare_amount') < 500,
                  "Rows removed where fare_amount >= 500")

df = apply_filter(df, pl.col('tip_amount') >= 0,
                  "Rows removed where tip_amount < 0")

df = apply_filter(df, pl.col('total_amount') > 0,
                  "Rows removed where total_amount <= 0")

df = apply_filter(
    df,
    pl.col("tpep_dropoff_datetime") >= pl.col("tpep_pickup_datetime"),
    "Rows removed where dropoff before pickup"
)

# --- Payment Type Cleaning ---

# Count how many will be changed
invalid_payment_rows = df.filter(
    (pl.col("payment_type") < 0) | (pl.col("payment_type") > 6)
).height

df = df.with_columns(
    pl.when((pl.col("payment_type") < 0) | (pl.col("payment_type") > 6))
      .then(5)
      .otherwise(pl.col("payment_type"))
      .alias("payment_type")
)

df = apply_filter(df, pl.col('payment_type') == 1,"Using only rows with Credit Card Payments")

print(f"Payment type values corrected: {invalid_payment_rows:,}")
print(f"Final number of rows in the dataframe: {df.height:,}")

In [ ]:
invalid_payment = df.filter((pl.col('payment_type') < 0) | (pl.col('payment_type') > 5)).sample(n=10, with_replacement=True)
print(f"Number of invalid payment records: {len(invalid_payment)}")

invalid_pc = df.filter(pl.col('passenger_count') <= 0).sample(n=10, with_replacement=True)
print(f"Records with passenger_count <= 0: {len(invalid_pc)}")

invalid_td = df.filter(pl.col('trip_distance') <= 0).sample(n=10, with_replacement=True)
print(f"Records with trip_distance <= 0: {len(invalid_td)}")

invalid_fa = df.filter(pl.col('fare_amount') <= 0).sample(n=10, with_replacement=True)
print(f"Records with fare_amount <= 0: {len(invalid_fa)}")

invalid_ta = df.filter(pl.col('tip_amount') < 0).sample(n=10, with_replacement=True)
print(f"Records with tip_amount < 0: {len(invalid_ta)}")

invalid_total = df.filter(pl.col('total_amount') <= 0).sample(n=10, with_replacement=True)
print(f"Records with total_amount <= 0: {len(invalid_total)}")


## Part 1: Data Preprocessing & Feature Engineering

In [ ]:
#Temporal Features
## pickup_hour, pickup_day_of_week (numeric, 0=Monday), is_weekend (boolean)
weekday_map = {
    0: "Monday", 1: "Tuesday", 2: "Wednesday", 3: "Thursday",
    4: "Friday", 5: "Saturday", 6: "Sunday"
}
df = df.with_columns([
    pl.col('tpep_pickup_datetime').dt.hour().alias('pickup_hour'),

    pl.col('tpep_pickup_datetime')
        .dt.weekday()
        .map_elements(lambda x: weekday_map.get(x, "Unknown"), return_dtype=pl.Utf8)
        .alias('pickup_day_of_week'),

    pl.col('tpep_pickup_datetime') 
        .dt.weekday()
        .is_in([5,6])
        .alias("is_weekend")   
])
#Trip features
##: trip_duration_minutes, trip_speed_mph, log_trip_distance (log transformed distance) 

duration_hours = (
    (pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
    .dt.total_seconds() / 3600
)

duration_minutes = duration_hours * 60

df = df.with_columns([
    ((pl.col('tpep_dropoff_datetime') - pl.col('tpep_pickup_datetime'))
        .dt.total_seconds() / 60).alias('trip_duration_minutes'),

    (pl.when(duration_hours > 0)
        .then(pl.col("trip_distance") / duration_hours)
        .otherwise(0)
        .clip(0, 80)
    ).alias("trip_speed_mph"),


    pl.col('trip_distance')
        .log()
        .alias('log_trip_distance')

])


#Fare features
##  fare_per_mile (fare_amount / trip_distance, handle division by zero), fare_per_minute (fare_amount / trip_duration_minutes) 
df = df.with_columns([
    (pl.col('fare_amount') / pl.col('trip_distance')) #already removed all 0 and negative trip distances 
        .alias('fare_per_mile'),

    (pl.when(pl.col('trip_duration_minutes') > 0)
        .then(pl.col('fare_amount') / pl.col('trip_duration_minutes'))
        .otherwise('fare_amount')
        ).alias('fare_per_minute')
])

#Zone features
## Encode pickup and dropoff borough using the taxi zone lookup table (use one-hot encoding or label encoding) 
zone_lookup = pl.read_csv("data/raw/taxi_zone_lookup.csv")
pickup_lookup = zone_lookup.select([
    pl.col('LocationID'),
    pl.col('Borough').alias('pickup_borough')
])

dropoff_lookup = zone_lookup.select([
    pl.col('LocationID'),
    pl.col('Borough').alias('dropoff_borough')
])

df = df.join(
    pickup_lookup,
    left_on='PULocationID',
    right_on='LocationID',
    how='left'
)

df = df.join(
    dropoff_lookup,
    left_on='DOLocationID',
    right_on='LocationID',
    how='left'
)

df = df.with_columns([
    pl.col('pickup_borough').cast(pl.Categorical).to_physical().alias('pickup_borough_label'),
    pl.col('dropoff_borough').cast(pl.Categorical).to_physical().alias('dropoff_borough_label')
])

In [ ]:
df.select(['pickup_hour','pickup_day_of_week',  'is_weekend', 'trip_duration_minutes','trip_speed_mph','log_trip_distance','fare_per_mile','fare_per_minute','PULocationID','pickup_borough','pickup_borough_label','DOLocationID','dropoff_borough','dropoff_borough_label']).head()


In [ ]:
df.schema

## Target Variable Generation, reusing existing tip_amount in the future

In [ ]:
df = df.with_columns([

    ((pl.col("tip_amount") > pl.col("fare_amount") * 0.20)
        .cast(pl.Int8)
    ).alias("high_tip")
])

Setting up the Splits for training

In [ ]:
from sklearn.model_selection import train_test_split

df_pd = df.to_pandas()

train_df, temp_df = train_test_split(
    df_pd,
    test_size=0.30,
    stratify=df_pd["high_tip"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["high_tip"],
    random_state=42
)

Number of samples in the splits as well as the class distribution for `high_tip`

In [ ]:
print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

In [ ]:
print("Train class distribution:")
print(train_df["high_tip"].value_counts())
print(train_df["high_tip"].value_counts(normalize=True))

In [ ]:
print("Validation class distribution:")
print(val_df["high_tip"].value_counts())
print(val_df["high_tip"].value_counts(normalize=True))

In [ ]:
print("Test class distribution:")
print(test_df["high_tip"].value_counts())
print(test_df["high_tip"].value_counts(normalize=True))

In [ ]:
X_train = train_df.drop(columns=["tip_amount", "high_tip"])


y_train_reg = train_df["tip_amount"]


y_train_clf = train_df["high_tip"]

Some numeric features were used, I left out any ID values, date values, classification type boolean values as well as values that are being targeted by the classification and regression.

In [ ]:
numeric_features = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "extra",
    "mta_tax",
    "tolls_amount",
    "improvement_surcharge",
    "congestion_surcharge",
    "Airport_fee",
    "pickup_hour",
    "trip_duration_minutes",
    "trip_speed_mph",
    "log_trip_distance"
]

print("Model Features:")
for col in numeric_features:
    print(f"{col} - numeric (scaled)")

print("\nExcluded Features:")
excluded = ["tip_amount", "high_tip",
            "VendorID", "PULocationID", "DOLocationID",
            "tpep_pickup_datetime", "tpep_dropoff_datetime",
            "pickup_borough", "dropoff_borough"]

for col in excluded:
    print(col)
#    

In [ ]:
from sklearn.preprocessing import StandardScaler



scaler = StandardScaler()

scaler.fit(train_df[numeric_features])

train_df[numeric_features] = scaler.transform(train_df[numeric_features])
val_df[numeric_features] = scaler.transform(val_df[numeric_features])
test_df[numeric_features] = scaler.transform(test_df[numeric_features])


train_df[numeric_features].describe()

## Part 2: Model Training & Tuning

### Baseline Model

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

feature_cols =[
    "passenger_count",
    "trip_distance",
    "log_trip_distance",
    "trip_duration_minutes",
    "trip_speed_mph",
    "pickup_hour",
    "is_weekend",
    "pickup_borough_label",
    "dropoff_borough_label",
    "fare_amount",
    "extra",
    "mta_tax",
    "tolls_amount",
    "congestion_surcharge",
    "Airport_fee",
    "fare_per_mile",
    "fare_per_minute"
]

# Features
X_train = train_df[feature_cols]
X_val = val_df[feature_cols]
X_test = test_df[feature_cols]

# Regression targets
y_train_reg = train_df["tip_amount"]
y_val_reg = val_df["tip_amount"]
y_test_reg = test_df["tip_amount"]

# Classification targets
y_train_clf = train_df["high_tip"]
y_val_clf = val_df["high_tip"]
y_test_clf = test_df["high_tip"]

### Regression

In [ ]:
linear_reg = LinearRegression()
linear_reg.fit(X_train, y_train_reg)

rf_reg = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_reg.fit(X_train, y_train_reg)

y_pred_lr_val = linear_reg.predict(X_val)
y_pred_rf_val = rf_reg.predict(X_val)




### Classification

In [ ]:
log_reg = LogisticRegression(
    random_state=42,
    max_iter=1000
)

log_reg.fit(X_train, y_train_clf)

rf_clf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_clf.fit(X_train, y_train_clf)

y_pred_log_val = log_reg.predict(X_val)
y_pred_rf_val = rf_clf.predict(X_val)


### Evaluation

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

import numpy as np
import pandas as pd

Regression

In [ ]:
regression_results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest Regressor"],
    "MAE": [
        mean_absolute_error(y_val_reg, y_pred_lr_val),
        mean_absolute_error(y_val_reg, y_pred_rf_val)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val_reg, y_pred_lr_val)),
        np.sqrt(mean_squared_error(y_val_reg, y_pred_rf_val))
    ],
    "R²": [
        r2_score(y_val_reg, y_pred_lr_val),
        r2_score(y_val_reg, y_pred_rf_val)
    ]
})

print("\n================ REGRESSION MODELS (Validation Set) ================\n")
print(regression_results.round(4))

After running the about 5 times and getting a different value on the second time, both times the Linear Regression performs better than the Random Forest in each metric, lower error values and a higher r2

Classification

In [ ]:
y_prob_log_val = log_reg.predict_proba(X_val)[:,1]
y_prob_rf_val = rf_clf.predict_proba(X_val)[:,1]

classification_results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest Classifier"],
    "Accuracy": [
        accuracy_score(y_val_clf, y_pred_log_val),
        accuracy_score(y_val_clf, y_pred_rf_val)
    ],
    "Precision": [
        precision_score(y_val_clf, y_pred_log_val),
        precision_score(y_val_clf, y_pred_rf_val)
    ],
    "Recall": [
        recall_score(y_val_clf, y_pred_log_val),
        recall_score(y_val_clf, y_pred_rf_val)
    ],
    "F1-score": [
        f1_score(y_val_clf, y_pred_log_val),
        f1_score(y_val_clf, y_pred_rf_val)
    ],
    "AUC-ROC": [
        roc_auc_score(y_val_clf, y_prob_log_val),
        roc_auc_score(y_val_clf, y_prob_rf_val)
    ]
})

print("\n================ CLASSIFICATION MODELS (Validation Set) ================\n")
print(classification_results.round(4))

Ran the fitting 3 times, unlike the regression, the values never changed, Logistic regression has a higher value in each metric aside from having a slightly lower precision.

### HyperParameter Tuning

Best Models:  
Linear Regression for Regression and Logistic Regression for Classification

In [ ]:
X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train,
    y_train_clf,
    train_size=300_000,
    stratify=y_train_clf,
    random_state=42
)

param_dist = {
    "C": [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver": ["liblinear", "saga"],
    "class_weight": [None, "balanced"]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

log_reg_tuned = LogisticRegression(
    max_iter=1000,
    random_state=42
)

random_search = RandomizedSearchCV(
    estimator=log_reg_tuned,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1,
    verbose=2
)

In [ ]:
random_search.fit(X_train_sample, y_train_sample) 

In [ ]:
print("===== Hyperparameter Search Space =====")
print(param_dist)

print("\n===== Best Parameters Found =====")
print(random_search.best_params_)

print("\n===== Best Cross-Validation Score =====")
print(random_search.best_score_)

In [ ]:
best_model = random_search.best_estimator_
y_pred_tuned_val = best_model.predict(X_val)
y_prob_tuned_val = best_model.predict_proba(X_val)[:, 1]

In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Baseline Logistic Regression", "Tuned Logistic Regression"],
    "Accuracy": [
        accuracy_score(y_val_clf, y_pred_log_val),
        accuracy_score(y_val_clf, y_pred_tuned_val)
    ],
    "Precision": [
        precision_score(y_val_clf, y_pred_log_val),
        precision_score(y_val_clf, y_pred_tuned_val)
    ],
    "Recall": [
        recall_score(y_val_clf, y_pred_log_val),
        recall_score(y_val_clf, y_pred_tuned_val)
    ],
    "F1-score": [
        f1_score(y_val_clf, y_pred_log_val),
        f1_score(y_val_clf, y_pred_tuned_val)
    ],
    "AUC-ROC": [
        roc_auc_score(y_val_clf, y_prob_log_val),
        roc_auc_score(y_val_clf, y_prob_tuned_val)
    ]
})

print("\n===== Validation Performance Comparison =====")
print(comparison_df.round(4))

In [ ]:
param_dist_lr = {
    "fit_intercept": [True, False],
    "positive": [True, False]
}

X_train_sample_reg, _, y_train_sample_reg, _ = train_test_split(
    X_train,
    y_train_reg,
    train_size=300_000,
    random_state=42
)


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RandomizedSearchCV

lr = LinearRegression()

random_search_lr = RandomizedSearchCV(
    estimator=lr,
    param_distributions=param_dist_lr,
    n_iter=4,
    cv=5,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=2
)

In [ ]:
random_search_lr.fit(X_train_sample_reg, y_train_sample_reg)

In [ ]:
print("Best parameters:")
print(random_search_lr.best_params_)

print("Best CV score:")
print(random_search_lr.best_score_)

In [ ]:
best_lr = random_search_lr.best_estimator_
y_pred_tuned_lr_val = best_lr.predict(X_val)

comparison_reg = pd.DataFrame({
    "Model": ["Baseline Linear Regression", "Tuned Linear Regression"],
    "MAE": [
        mean_absolute_error(y_val_reg, y_pred_lr_val),
        mean_absolute_error(y_val_reg, y_pred_tuned_lr_val)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val_reg, y_pred_lr_val)),
        np.sqrt(mean_squared_error(y_val_reg, y_pred_tuned_lr_val))
    ],
    "R²": [
        r2_score(y_val_reg, y_pred_lr_val),
        r2_score(y_val_reg, y_pred_tuned_lr_val)
    ]
})

print(comparison_reg.round(4))

### Neural Network Model

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

X_train_nn = train_df[feature_cols].to_numpy(dtype=np.float32)
X_val_nn = val_df[feature_cols].to_numpy(dtype=np.float32)
X_test_nn = test_df[feature_cols].to_numpy(dtype=np.float32)

y_train_nn = train_df["high_tip"].to_numpy(dtype=np.float32).reshape(-1, 1)
y_val_nn = val_df["high_tip"].to_numpy(dtype=np.float32).reshape(-1, 1)
y_test_nn = test_df["high_tip"].to_numpy(dtype=np.float32).reshape(-1, 1)

train_dataset = TensorDataset(
    torch.tensor(X_train_nn),
    torch.tensor(y_train_nn)
)

val_dataset = TensorDataset(
    torch.tensor(X_val_nn),
    torch.tensor(y_val_nn)
)

test_dataset = TensorDataset(
    torch.tensor(X_test_nn),
    torch.tensor(y_test_nn)
)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

In [ ]:
import torch.nn as nn

class TipClassifierNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.network(x)

In [ ]:
input_dim = X_train_nn.shape[1]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TipClassifierNN(input_dim).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
num_epochs = 20

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    model.train()
    running_train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * X_batch.size(0)

    epoch_train_loss = running_train_loss / len(train_loader.dataset)
    train_losses.append(epoch_train_loss)

    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            running_val_loss += loss.item() * X_batch.size(0)

    epoch_val_loss = running_val_loss / len(val_loader.dataset)
    val_losses.append(epoch_val_loss)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f}"
    )

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, num_epochs + 1), train_losses, label="Training Loss")
plt.plot(range(1, num_epochs + 1), val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

model.eval()

val_probs = []
val_preds = []
val_true = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)

        logits = model(X_batch)
        probs = torch.sigmoid(logits)

        preds = (probs >= 0.5).float()

        val_probs.extend(probs.cpu().numpy().flatten())
        val_preds.extend(preds.cpu().numpy().flatten())
        val_true.extend(y_batch.numpy().flatten())

In [ ]:
import pandas as pd
nn_classification_results = pd.DataFrame({
    "Model": ["Neural Network Classifier"],
    "Accuracy": [accuracy_score(val_true, val_preds)],
    "Precision": [precision_score(val_true, val_preds)],
    "Recall": [recall_score(val_true, val_preds)],
    "F1-score": [f1_score(val_true, val_preds)],
    "AUC-ROC": [roc_auc_score(val_true, val_probs)]
})

print(nn_classification_results.round(4))

In [ ]:
comparison_nn_clf = pd.DataFrame({
    "Model": [
        "Baseline Logistic Regression",
        "Tuned Logistic Regression",
        "Neural Network Classifier"
    ],
    "Accuracy": [
        accuracy_score(y_val_clf, y_pred_log_val),
        accuracy_score(y_val_clf, y_pred_tuned_val),
        accuracy_score(val_true, val_preds)
    ],
    "Precision": [
        precision_score(y_val_clf, y_pred_log_val),
        precision_score(y_val_clf, y_pred_tuned_val),
        precision_score(val_true, val_preds)
    ],
    "Recall": [
        recall_score(y_val_clf, y_pred_log_val),
        recall_score(y_val_clf, y_pred_tuned_val),
        recall_score(val_true, val_preds)
    ],
    "F1-score": [
        f1_score(y_val_clf, y_pred_log_val),
        f1_score(y_val_clf, y_pred_tuned_val),
        f1_score(val_true, val_preds)
    ],
    "AUC-ROC": [
        roc_auc_score(y_val_clf, y_prob_log_val),
        roc_auc_score(y_val_clf, y_prob_tuned_val),
        roc_auc_score(val_true, val_probs)
    ]
})

print(comparison_nn_clf.round(4))

In [ ]:
model.eval()

test_probs = []
test_preds = []
test_true = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)

        logits = model(X_batch)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()

        test_probs.extend(probs.cpu().numpy().flatten())
        test_preds.extend(preds.cpu().numpy().flatten())
        test_true.extend(y_batch.numpy().flatten())

print("Neural Network Test Metrics")
print("Accuracy:", round(accuracy_score(test_true, test_preds), 4))
print("Precision:", round(precision_score(test_true, test_preds), 4))
print("Recall:", round(recall_score(test_true, test_preds), 4))
print("F1-score:", round(f1_score(test_true, test_preds), 4))
print("AUC-ROC:", round(roc_auc_score(test_true, test_probs), 4))

## Part 3: Model Evaluation & Interpretation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# =========================================================
# FINAL MODEL COMPARISON
# =========================================================

# -------------------------
# Regression Model Summary
# -------------------------
regression_summary = pd.DataFrame({
    "Model": [
        "Linear Regression (Baseline)",
        "Random Forest Regressor (Baseline)",
        "Linear Regression (Tuned)"
    ],
    "MAE": [
        mean_absolute_error(y_val_reg, y_pred_lr_val),
        mean_absolute_error(y_val_reg, y_pred_rf_val),
        mean_absolute_error(y_val_reg, y_pred_tuned_lr_val)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_val_reg, y_pred_lr_val)),
        np.sqrt(mean_squared_error(y_val_reg, y_pred_rf_val)),
        np.sqrt(mean_squared_error(y_val_reg, y_pred_tuned_lr_val))
    ],
    "R²": [
        r2_score(y_val_reg, y_pred_lr_val),
        r2_score(y_val_reg, y_pred_rf_val),
        r2_score(y_val_reg, y_pred_tuned_lr_val)
    ]
})

# -----------------------------
# Classification Model Summary
# -----------------------------
classification_summary = pd.DataFrame({
    "Model": [
        "Logistic Regression (Baseline)",
        "Random Forest Classifier (Baseline)",
        "Logistic Regression (Tuned)",
        "Neural Network Classifier"
    ],
    "Accuracy": [
        accuracy_score(y_val_clf, y_pred_log_val),
        accuracy_score(y_val_clf, y_pred_rf_val),
        accuracy_score(y_val_clf, y_pred_tuned_val),
        accuracy_score(val_true, val_preds)
    ],
    "Precision": [
        precision_score(y_val_clf, y_pred_log_val),
        precision_score(y_val_clf, y_pred_rf_val),
        precision_score(y_val_clf, y_pred_tuned_val),
        precision_score(val_true, val_preds)
    ],
    "Recall": [
        recall_score(y_val_clf, y_pred_log_val),
        recall_score(y_val_clf, y_pred_rf_val),
        recall_score(y_val_clf, y_pred_tuned_val),
        recall_score(val_true, val_preds)
    ],
    "F1-score": [
        f1_score(y_val_clf, y_pred_log_val),
        f1_score(y_val_clf, y_pred_rf_val),
        f1_score(y_val_clf, y_pred_tuned_val),
        f1_score(val_true, val_preds)
    ],
    "AUC-ROC": [
        roc_auc_score(y_val_clf, y_prob_log_val),
        roc_auc_score(y_val_clf, y_prob_rf_val),
        roc_auc_score(y_val_clf, y_prob_tuned_val),
        roc_auc_score(val_true, val_probs)
    ]
})

print("=========================================================")
print("REGRESSION MODEL PERFORMANCE (VALIDATION SET)")
print("=========================================================")
print(regression_summary.round(4).to_string(index=False))

print("\n=========================================================")
print("CLASSIFICATION MODEL PERFORMANCE (VALIDATION SET)")
print("=========================================================")
print(classification_summary.round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

# Convert to plain arrays if needed
y_true = y_val_clf.to_numpy() if hasattr(y_val_clf, "to_numpy") else y_val_clf

models_probs = {
    "Logistic Regression (Baseline)": y_prob_log_val,
    "Random Forest Classifier (Baseline)": y_prob_rf_val,
    "Logistic Regression (Tuned)": y_prob_tuned_val,
    "Neural Network Classifier": val_probs
}

plt.figure(figsize=(8, 6))

for model_name, y_prob in models_probs.items():
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = roc_auc_score(y_true, y_prob)

    plt.plot(
        fpr,
        tpr,
        label=f"{model_name} (AUC = {auc_score:.4f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for Classification Models")
plt.legend()
plt.grid(True)
plt.show()

## At the time of doing this the Neural Network was my best performing Classification Model

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

cm = confusion_matrix(val_true, val_preds)

labels = np.array([
    ["True Negative", "False Positive"],
    ["False Negative", "True Positive"]
])

annot = np.empty_like(labels, dtype=object)
for i in range(2):
    for j in range(2):
        annot[i, j] = f"{labels[i, j]}\n{cm[i, j]}"

plt.figure(figsize=(6, 6))
sns.heatmap(
    cm,
    annot=annot,
    fmt="",
    cmap="Blues",
    cbar=False,
    xticklabels=["Not High Tip", "High Tip"],
    yticklabels=["Not High Tip", "High Tip"]
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix: Neural Network Classifier")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


y_true = y_val_reg


y_pred = y_pred_lr_val

plt.figure(figsize=(7,7))

plt.scatter(
    y_true,
    y_pred,
    alpha=0.3
)


max_val = max(y_true.max(), y_pred.max())

plt.plot(
    [0, max_val],
    [0, max_val],
    color="red",
    linestyle="--",
    label="Perfect Prediction"
)

plt.xlabel("Actual Tip Amount ($)")
plt.ylabel("Predicted Tip Amount ($)")
plt.title("Predicted vs Actual Tip Amounts (Linear Regression)")
plt.legend()
plt.grid(True)
plt.xlim(0, 50)
plt.ylim(0, 50)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# Compute residuals
# -----------------------------
y_true = y_val_reg
y_pred = y_pred_lr_val

residuals = y_true - y_pred

# -----------------------------
# Create residual analysis plots
# -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Residual Distribution
axes[0].hist(residuals, bins=50)
axes[0].set_title("Residual Distribution (Linear Regression)")
axes[0].set_xlabel("Residual (Actual - Predicted)")
axes[0].set_ylabel("Count")
axes[0].axvline(0, linestyle="--")  # ideal center line

# Residuals vs Predicted
axes[1].scatter(y_pred, residuals, alpha=0.3)
axes[1].axhline(0, linestyle="--")  # ideal residual line
axes[1].set_title("Residuals vs Predicted Tip Amount")
axes[1].set_xlabel("Predicted Tip Amount ($)")
axes[1].set_ylabel("Residual (Actual - Predicted)")
axes[1].set_ylim(-20, 20)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

rf_importances = rf_reg.feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_importances
})

feature_importance_df = feature_importance_df.sort_values(
    by="Importance",
    ascending=False
)

plt.figure(figsize=(10,6))

plt.barh(
    feature_importance_df["Feature"],
    feature_importance_df["Importance"]
)

plt.gca().invert_yaxis()

plt.title("Random Forest Feature Importance")
plt.xlabel("Importance Score")
plt.ylabel("Feature")

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

lr_coefficients = linear_reg.coef_

lr_coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": lr_coefficients
})

lr_coef_df = lr_coef_df.sort_values(
    by="Coefficient",
    key=abs,
    ascending=False
)

print("Linear Regression Coefficients")
print(lr_coef_df)

plt.figure(figsize=(10, 6))
plt.barh(
    lr_coef_df["Feature"],
    lr_coef_df["Coefficient"]
)
plt.gca().invert_yaxis()
plt.title("Linear Regression Feature Coefficients")
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.show()

In [ ]:
log_coefficients = log_reg.coef_[0]

log_coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": log_coefficients
})

log_coef_df = log_coef_df.sort_values(
    by="Coefficient",
    key=abs,
    ascending=False
)

print("Logistic Regression Coefficients")
print(log_coef_df)

plt.figure(figsize=(10, 6))
plt.barh(
    log_coef_df["Feature"],
    log_coef_df["Coefficient"]
)
plt.gca().invert_yaxis()
plt.title("Logistic Regression Feature Coefficients")
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.show()

## Written Analysis

### Most Predictive features
Unsurprisingly fare amount is the most predicitive feature for tip amount with regards to determining the tip amount, strangely trip_distance and trip_duration were not very high. Given that tips are usually based on how well a service is performed, and intuitively one would reward a better service, so the faster you get to your destination you would tip more. As well as the fact that fare amount would correlate strongly with fare_amount since they charge I believe based on how far they take you and America tends to tip a percentage of what they are charged.

### Limitations of the models
Dataset bias is a possibility since payments had to be restricted purely to credit card transactiosn to get accurate tip values. Cash transaction may provide different tipping behaviour, thus models may not generalize to all taxi trips. The dataset was also located specifically in NYC during the month of January, models trained on this data may not generalize well to other cities and time periods as is.




## AI Disclosure
Used AI to help generate the evaluation outputs so that they're neat and readable